# Sujet 3 — Notebook 2 : règles et alertes expliquées

**Architecture hybride.**

| Brique | Rôle | Fichier |
|---|---|---|
| Graphe de connaissances (Hetionet) | faits : classes (PCiC), cibles (CbG), effets (CcSE) | `src/graphe.py` |
| Connaissances expertes | synonymes, table ATC complémentaire, effets graves, marge étroite | `src/connaissances.py` |
| Raisonnement par règles | 18 règles expertes + 3 règles génériques sur le graphe | `src/regles.py` |

Chaque alerte donne : son **niveau** (échelle du Thésaurus ANSM), la **règle** déclenchée, sa **source**, et le **chemin dans le graphe** (ou la ligne de table) qui la justifie.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Les modules sont dans src/. Sur Colab : importez sujet3_projet.zip puis exécutez
# !unzip -q sujet3_projet.zip   (le dossier projet/ apparaît à côté du notebook)
ICI = Path.cwd()
for c in [ICI, ICI / "src", ICI.parent / "src", ICI / "projet" / "src"]:
    if (c / "graphe.py").exists():
        sys.path.insert(0, str(c)); break

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

from graphe import charger_graphe
from regles import Verificateur, NIVEAUX
from evaluation import tableau_regles

G = charger_graphe()
V = Verificateur(G)   # réglages par défaut : mode concis, filtre « marge étroite » sur G1

## 1. Les règles

Niveaux : 4 = contre-indication · 3 = association déconseillée · 2 = précaution d'emploi · 1 = à prendre en compte.

In [2]:
display(tableau_regles())

,code,type,condition,niveau,risque,source
0,R01,classe (experte),AVK + AINS,Association déconseillée,majoration du risque hémorragique,"ANSM, Thésaurus des interactions médicamenteuses : AVK + AINS"
1,R02,classe (experte),ANTICOAGULANT + ANTIAGREGANT,À prendre en compte,majoration du risque hémorragique,"ANSM, Thésaurus des interactions médicamenteuses : anticoagulants oraux + antiagrégants plaquettaires"
2,R03,classe (experte),ANTICOAGULANT + ANTICOAGULANT,Contre-indication,risque hémorragique majeur (hors relais),"RCP Eliquis / Xarelto (EMA), rubrique 4.3 contre-indications"
3,R04,classe (experte),STATINE_3A4 + INHIB_3A4_FORT,Contre-indication,rhabdomyolyse,"ANSM, Thésaurus des interactions médicamenteuses : simvastatine + inhibiteurs puissants du CYP3A4 ; RCP Zocor"
4,R05,classe (experte),NITRE + IPDE5,Contre-indication,hypotension sévère,"ANSM, Thésaurus des interactions médicamenteuses ; RCP Viagra (EMA), rubrique 4.3"
5,R06,classe (experte),IMAO_NS + SEROTONINERGIQUE_LARGE,Contre-indication,syndrome sérotoninergique,"ANSM, Thésaurus des interactions médicamenteuses : IMAO non sélectifs + ISRS"
6,R07,classe (experte),ISRS + SEROTONINERGIQUE,À prendre en compte,"syndrome sérotoninergique, convulsions","ANSM, Thésaurus des interactions médicamenteuses : médicaments sérotoninergiques ; tramadol + ISRS"
7,R08,classe (experte),SRAA + EPARGNEUR_K,Association déconseillée,hyperkaliémie,"ANSM, Thésaurus des interactions médicamenteuses : épargneurs de potassium + IEC / ARA II"
8,R09,classe (experte),METHOTREXATE + AINS,Association déconseillée,toxicité hématologique du méthotrexate (doses > 20 mg/semaine),"ANSM, Thésaurus des interactions médicamenteuses : méthotrexate + AINS"
9,R10,classe (experte),METHOTREXATE + ANTIFOLIQUE,Association déconseillée,toxicité hématologique,"ANSM, Thésaurus des interactions médicamenteuses : méthotrexate + triméthoprime"


## 2. Trois façons de déclencher une alerte

- **Règle par classe** (R01-R18) : les deux médicaments appartiennent à deux classes connues pour interagir. L'appartenance vient du graphe (arête PCiC) ou de la table ATC.
- **G1 pharmacocinétique** : A est classé « inhibiteur (ou inducteur) d'une enzyme » dans le graphe et B est lié à cette enzyme (CbG). Niveau 2 seulement si B est à **marge thérapeutique étroite** ; sinon niveau 1.
- **G2 / G3** : effet indésirable grave partagé (CcSE) ou cible pharmacologique commune (CbG, hors enzymes et transporteurs). Signaux faibles : niveau 1.

## 3. Exemples

### Le cas d'école : warfarine + ibuprofène

In [3]:
print(V.rapport(["warfarine", "ibuprofène", "oméprazole"]))

Ordonnance : warfarine (Warfarin), ibuprofène (Ibuprofen), oméprazole (Omeprazole)
[Association déconseillée] warfarine + ibuprofène
   • R01 (Association déconseillée) — majoration du risque hémorragique
       Warfarin ←[PCiC]— « Vitamin K Inhibitors » ⇒ antivitamines K
       Ibuprofen ←[PCiC]— « Nonsteroidal Anti-inflammatory Compounds » ⇒ anti-inflammatoires non stéroïdiens
       Source : ANSM, Thésaurus des interactions médicamenteuses : AVK + AINS
[Précaution d'emploi] warfarine + oméprazole
   • G1 (Précaution d'emploi) — modification des concentrations (pharmacocinétique)
       Omeprazole ←[PCiC]— « Cytochrome P450 2C19 Inhibitors » ; Warfarin —[CbG]→ CYP2C19 ⇒ Omeprazole inhibe l'élimination de Warfarin
       Warfarin : marge thérapeutique étroite ⇒ niveau relevé
       Source : Hetionet : classe d'inhibiteur/inducteur (PCiC) + liaison à l'enzyme (CbG) ; niveau 2 si la victime est à marge thérapeutique étroite, sinon 1
(2 alerte(s), 1.0 ms)


### Contre-indication détectée deux fois : par la règle experte et par le graphe

In [4]:
print(V.rapport(["simvastatine", "clarithromycine", "amlodipine"]))

Ordonnance : simvastatine (Simvastatin), clarithromycine (Clarithromycin), amlodipine
[Contre-indication] simvastatine + clarithromycine
   • R04 (Contre-indication) — rhabdomyolyse
       Simvastatin ∈ statines métabolisées par le CYP3A4 (table ATC C10AA01)
       Clarithromycin ∈ inhibiteurs puissants du CYP3A4 (table ATC J01FA09)
       Source : ANSM, Thésaurus des interactions médicamenteuses : simvastatine + inhibiteurs puissants du CYP3A4 ; RCP Zocor
(1 alerte(s), 0.8 ms)


### Médicament absent du graphe et noms commerciaux

La fluindione (Previscan) n'existe pas dans Hetionet : seule la table ATC permet de la classer.

In [5]:
print(V.rapport(["Previscan", "Advil"]))

Ordonnance : Previscan (Fluindione), Advil (Ibuprofen)
⚠ Fluindione absent du graphe : seules les règles par classe (table) s'appliquent.
[Association déconseillée] Previscan + Advil
   • R01 (Association déconseillée) — majoration du risque hémorragique
       Fluindione ∈ antivitamines K (table ATC B01AA12)
       Ibuprofen ←[PCiC]— « Nonsteroidal Anti-inflammatory Compounds » ⇒ anti-inflammatoires non stéroïdiens
       Source : ANSM, Thésaurus des interactions médicamenteuses : AVK + AINS
(1 alerte(s), 0.2 ms)


### Interaction qu'aucune règle experte ne couvre : seul le graphe la trouve

In [6]:
print(V.rapport(["colchicine", "clarithromycine"]))

Ordonnance : colchicine, clarithromycine (Clarithromycin)
[Précaution d'emploi] colchicine + clarithromycine
   • G1 (Précaution d'emploi) — modification des concentrations (pharmacocinétique)
       Colchicine ←[PCiC]— « Cytochrome P450 3A4 Inhibitors » ; Clarithromycin —[CbG]→ CYP3A4 ⇒ Colchicine inhibe l'élimination de Clarithromycin
       Clarithromycin ←[PCiC]— « Cytochrome P450 3A4 Inhibitors » ; Colchicine —[CbG]→ CYP3A4 ⇒ Clarithromycin inhibe l'élimination de Colchicine
       Clarithromycin ←[PCiC]— « P-Glycoprotein Inhibitors » ; Colchicine —[CbG]→ ABCB1 ⇒ Clarithromycin inhibe l'élimination de Colchicine
       Colchicine : marge thérapeutique étroite ⇒ niveau relevé
       Source : Hetionet : classe d'inhibiteur/inducteur (PCiC) + liaison à l'enzyme (CbG) ; niveau 2 si la victime est à marge thérapeutique étroite, sinon 1
(1 alerte(s), 0.5 ms)


### Saisie incertaine

Un nom mal orthographié est reconnu par correspondance approchée **et signalé** ; un nom inconnu n'est jamais ignoré en silence.

In [7]:
print(V.rapport(["warfarinne", "Advil", "médicament-inconnu"]))

Ordonnance : warfarinne (Warfarin), Advil (Ibuprofen), médicament-inconnu
⚠ « warfarinne » interprété comme Warfarin : à confirmer.
⚠ « médicament-inconnu » non reconnu : interactions NON vérifiées pour ce médicament.
[Association déconseillée] warfarinne + Advil
   • R01 (Association déconseillée) — majoration du risque hémorragique
       Warfarin ←[PCiC]— « Vitamin K Inhibitors » ⇒ antivitamines K
       Ibuprofen ←[PCiC]— « Nonsteroidal Anti-inflammatory Compounds » ⇒ anti-inflammatoires non stéroïdiens
       Source : ANSM, Thésaurus des interactions médicamenteuses : AVK + AINS
(1 alerte(s), 4.9 ms)


### Fatigue d'alerte : mode concis contre mode complet

Ordonnance de 8 médicaments d'un patient âgé, sans interaction attendue.

In [8]:
O18 = ["bisoprolol", "ramipril", "atorvastatine", "Kardégic", "metformine", "oméprazole", "paracétamol", "Lévothyrox"]
complet = Verificateur(G, concis=False)
print("Mode complet :", len(complet.verifier(O18)[0]), "alertes")
print("Mode concis  :", len(V.verifier(O18)[0]), "alerte(s)\n")
print(V.rapport(O18))

Mode complet : 22 alertes
Mode concis  : 1 alerte(s)

Ordonnance : bisoprolol, ramipril, atorvastatine (Atorvastatin), Kardégic (Acetylsalicylic acid), metformine (Metformin), oméprazole (Omeprazole), paracétamol (Acetaminophen), Lévothyrox (Levothyroxine)
[Précaution d'emploi] ramipril + Kardégic
   • R16 (Précaution d'emploi) — insuffisance rénale aiguë
       Ramipril ∈ inhibiteurs de l'enzyme de conversion (table ATC C09AA05)
       Acetylsalicylic acid ←[PCiC]— « Nonsteroidal Anti-inflammatory Compounds » ⇒ anti-inflammatoires non stéroïdiens
       Source : ANSM, Thésaurus des interactions médicamenteuses : AINS + IEC / ARA II
(1 alerte(s), 4.7 ms)


**Lecture.** En mode complet, la règle G2 (effet indésirable partagé) s'allume sur presque toutes les paires : les listes d'effets de SIDER sont longues et sans fréquence. Le mode concis ne garde que les règles expertes et les signaux du graphe de niveau ≥ 2.

L'alerte restante (aspirine + ramipril, R16) est un **faux positif dû au graphe** : Hetionet classe l'aspirine parmi les AINS, alors qu'à dose antiagrégante (Kardégic) elle n'est pas concernée par l'interaction AINS + IEC. Le graphe ne connaît pas les doses.